In [14]:
# MLP Module
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
batch_size = 32

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.MLP = nn.Sequential(
            nn.Flatten(),
            nn.Linear(9, 140),
            nn.ReLU(),
            nn.Linear(140, 140),
            nn.ReLU(),
            nn.Linear(140, 140),
            nn.ReLU(),
            nn.Linear(140, 140),
            nn.ReLU(),
            nn.Linear(140, 1)
        )
    def forward(self, X):
        return self.MLP(X)

from torchsummary import summary
model = MLP()
summary(model, input_size=(9,1), device='cpu')

learning_rate= 0.00015

loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),eps= 1e-8,lr= learning_rate)
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)  
        if m.bias is not None:
            nn.init.zeros_(m.bias)  

model.apply(init_weights)




cuda
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                    [-1, 9]               0
            Linear-2                  [-1, 140]           1,400
              ReLU-3                  [-1, 140]               0
            Linear-4                  [-1, 140]          19,740
              ReLU-5                  [-1, 140]               0
            Linear-6                  [-1, 140]          19,740
              ReLU-7                  [-1, 140]               0
            Linear-8                  [-1, 140]          19,740
              ReLU-9                  [-1, 140]               0
           Linear-10                    [-1, 1]             141
Total params: 60,761
Trainable params: 60,761
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 0.23
Estimate

MLP(
  (MLP): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=9, out_features=140, bias=True)
    (2): ReLU()
    (3): Linear(in_features=140, out_features=140, bias=True)
    (4): ReLU()
    (5): Linear(in_features=140, out_features=140, bias=True)
    (6): ReLU()
    (7): Linear(in_features=140, out_features=140, bias=True)
    (8): ReLU()
    (9): Linear(in_features=140, out_features=1, bias=True)
  )
)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [16]:

import h5py
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

def build_dataloaders_from_h5(
    h5_path: str,
    batch_size: int = 32,          
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
    num_workers: int = 0,
    zscore_normalize: bool = True,
    eps: float = 1e-8
):

    s = train_ratio + val_ratio + test_ratio
    if abs(s - 1.0) > 1e-12:
        raise ValueError(f"train_ratio + val_ratio + test_ratio must equal 1, current value is {s}")
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())
        if "X" not in f or "Y" not in f:
            raise KeyError(f"HDF5 file must contain keys 'X' and 'Y', current keys are: {keys}")

        X_np = f["X"][:]   
        Y_np = f["Y"][:]   

    X = torch.tensor(X_np, dtype=torch.float32)
    Y = torch.tensor(Y_np, dtype=torch.float32)

    if Y.ndim == 1:
        Y = Y.unsqueeze(1)

    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"Number of samples in X and Y are inconsistent: {X.shape[0]} vs {Y.shape[0]}")

    N = X.shape[0]
    if N < 10:
        raise ValueError(f"Too few samples N={N}, not suitable for stable splitting and training.")

    full_dataset = TensorDataset(X, Y)

    n_train = int(N * train_ratio)
    n_val = int(N * val_ratio)
    n_test = N - n_train - n_val  

    if min(n_train, n_val, n_test) <= 0:
        raise ValueError(
            f"After splitting, some parts are empty: n_train={n_train}, n_val={n_val}, n_test={n_test}"
        )

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set = random_split(
        full_dataset, [n_train, n_val, n_test], generator=generator
    )

    mu = None
    sigma = None

    if zscore_normalize:
        train_indices = train_set.indices

        X_train_samples = X[train_indices]  
        mu = X_train_samples.mean(dim=0)
        sigma = X_train_samples.std(dim=0, unbiased=False)
        sigma = sigma.clone()
        sigma[sigma < eps] = 1.0

        X.sub_(mu).div_(sigma)

    #DataLoader 
    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,     # Shuffle training set
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False
    )

    val_loader = DataLoader(
        val_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False
    )

    test_loader = DataLoader(
        test_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False
    )

    meta = {
        "N_total": N,
        "N_train": n_train,
        "N_val": n_val,
        "N_test": n_test,
        "X_shape": tuple(X.shape),
        "Y_shape": tuple(Y.shape),
        "batch_size": batch_size,
        "seed": seed,
        "zscore_normalize": zscore_normalize,
        "mu": mu,         # Must be saved for subsequent inference
        "sigma": sigma,   # Must be saved for subsequent inference
        "eps": eps,
    }

    print("Data split summary:")
    print({
        "N_total": meta["N_total"],
        "N_train": meta["N_train"],
        "N_val": meta["N_val"],
        "N_test": meta["N_test"],
        "X_shape": meta["X_shape"],
        "Y_shape": meta["Y_shape"],
        "batch_size": meta["batch_size"],
        "zscore_normalize": meta["zscore_normalize"],
    })

    return train_loader, val_loader, test_loader, meta

train_loader, val_loader, test_loader, data_meta = build_dataloaders_from_h5(
    h5_path="data/train_rho266.h5",
    batch_size=32,    
    seed=42,
    num_workers=0
 )

Data split summary:
{'N_total': 3408020, 'N_train': 2385614, 'N_val': 511203, 'N_test': 511203, 'X_shape': (3408020, 9), 'Y_shape': (3408020, 1), 'batch_size': 32, 'zscore_normalize': True}


In [17]:

import copy
import math
import time
import torch
import torch.nn as nn


def train_one_epoch(model, loader, optimizer, criterion, device, grad_clip=None):
    model.train()

    total_loss = 0.0     
    total_mae = 0.0      
    total_count = 0
    max_ae = 0.0         

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        y_pred = model(X_batch)

        if y_pred.shape != y_batch.shape:
            y_batch = y_batch.view_as(y_pred)

    
        loss = criterion(y_pred, y_batch)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step()

        # Error statistics
        batch_ae = torch.abs(y_pred - y_batch)
        current_max_ae = torch.max(batch_ae).item()
        if current_max_ae > max_ae:
            max_ae = current_max_ae

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_mae += batch_ae.mean().item() * batch_size
        total_count += batch_size

    mse = total_loss / total_count
    rmse = math.sqrt(mse)
    mae = total_mae / total_count

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "max_ae": max_ae,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Validation/test set evaluation:
    - model.eval()
    - torch.no_grad()
    - return MSE / RMSE / MAE / MaxAE
    """
    model.eval()

    total_loss = 0.0
    total_mae = 0.0
    total_count = 0
    max_ae = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        y_pred = model(X_batch)

        if y_pred.shape != y_batch.shape:
            y_batch = y_batch.view_as(y_pred)

        loss = criterion(y_pred, y_batch)

        batch_ae = torch.abs(y_pred - y_batch)
        current_max_ae = torch.max(batch_ae).item()
        if current_max_ae > max_ae:
            max_ae = current_max_ae

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_mae += batch_ae.mean().item() * batch_size
        total_count += batch_size

    mse = total_loss / total_count
    rmse = math.sqrt(mse)
    mae = total_mae / total_count

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "max_ae": max_ae,
    }

def fit_regression_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    *,
    epochs=72,
    lr=1.5e-4,             
    patience=30,            
    grad_clip=None,
    save_path="model_rho266.pth",
    print_every=1,
    use_scheduler=False     #
):

    model = model.to(device)

    # If the paper explicitly specifies a loss function other than MSE, replace here
    criterion = nn.MSELoss()

    # Align with paper: Adam
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode="min", 
            factor=0.5, 
            patience=10, 
            verbose=True
        )

    best_val_mae = float("inf")
    best_epoch = -1
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    history = {
        "train_mse": [], "train_rmse": [], "train_mae": [], "train_max_ae": [],
        "val_mse": [],   "val_rmse": [],   "val_mae": [],   "val_max_ae": [],
        "lr": []
    }

    t0 = time.time()

    for epoch in range(1, epochs + 1):
        # -Train 
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            grad_clip=grad_clip
        )

        val_metrics = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device
        )

        if scheduler is not None:
            scheduler.step(val_metrics["mae"])

        current_lr = optimizer.param_groups[0]["lr"]

        # Record history
        history["train_mse"].append(train_metrics["mse"])
        history["train_rmse"].append(train_metrics["rmse"])
        history["train_mae"].append(train_metrics["mae"])
        history["train_max_ae"].append(train_metrics["max_ae"])

        history["val_mse"].append(val_metrics["mse"])
        history["val_rmse"].append(val_metrics["rmse"])
        history["val_mae"].append(val_metrics["mae"])
        history["val_max_ae"].append(val_metrics["max_ae"])

        history["lr"].append(current_lr)

        improved = val_metrics["mae"] < best_val_mae

        if improved:
            best_val_mae = val_metrics["mae"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

  
        if (epoch % print_every == 0) or improved or (epoch == 1):
            flag = " <- best(val MAE)" if improved else ""
            print(
                f"[Epoch {epoch:03d}/{epochs}] "
                f"LR={current_lr:.3e} | "
                f"Train: MSE={train_metrics['mse']:.6e}, RMSE={train_metrics['rmse']:.6e}, "
                f"MAE={train_metrics['mae']:.6e}, MaxAE={train_metrics['max_ae']:.6e} | "
                f"Val: MSE={val_metrics['mse']:.6e}, RMSE={val_metrics['rmse']:.6e}, "
                f"MAE={val_metrics['mae']:.6e}, MaxAE={val_metrics['max_ae']:.6e}"
                f"{flag}"
            )

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch} (patience={patience}).")
            break

    elapsed = time.time() - t0
    print(f"\nTraining finished in {elapsed:.2f}s")
    print(f"Best epoch (by val MAE): {best_epoch}")
    print(f"Best val MAE: {best_val_mae:.6e}")

    model.load_state_dict(best_state)

    test_metrics = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device
    )

    print("\nFinal TEST metrics (best checkpoint selected by val MAE):")
    print(f"Test MSE   = {test_metrics['mse']:.6e}")
    print(f"Test RMSE  = {test_metrics['rmse']:.6e}")
    print(f"Test MAE   = {test_metrics['mae']:.6e}")
    print(f"Test MaxAE = {test_metrics['max_ae']:.6e}")

    result = {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_mae": best_val_mae,
        "test_metrics": test_metrics,
        "save_path": save_path,
    }
    return result



model = MLP()

result = fit_regression_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=72,
    lr=1.5e-4,
     patience=30,
    grad_clip=None,              
    save_path="./model/model_rho266.pth",
     print_every=1,
    use_scheduler=False
 )

AcceleratorError: CUDA error: unspecified launch failure
Search for `cudaErrorLaunchFailure' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
